# 02. Machine Learning Model Comparison & Ensembling

This notebook compares candidate classification architectures under **Spatial Leave-One-Map-Out Cross Validation**:
1. Random Forest Classifier (Baseline)
2. HistGradientBoostingClassifier
3. Soft-Voting Ensemble (Random Forest + HistGradientBoosting)

**Strict Rule:** Cross-validation is strictly grouped by `map_id` to prevent spatial data leakage.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut

df = pd.read_csv('../train_cells_capped.csv')
base_feats = ['avg_speed_kmh', 'signal_delay_sec', 'traffic_density_0_1', 'intersection_density_0_1', 'incident_probability_0_1']
df_enc = pd.get_dummies(df, columns=['road_type'])
road_cols = [c for c in df_enc.columns if c.startswith('road_type_')]
all_feats = base_feats + road_cols

X = df_enc[all_feats]
y = df['congestion_label']
groups = df['map_id']


In [ ]:
# Train and evaluate candidate models using Leave-One-Map-Out
models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, max_depth=6, class_weight='balanced', random_state=42)
}

logo = LeaveOneGroupOut()
for name, clf in models.items():
    y_true, y_pred = [], []
    for tr_idx, te_idx in logo.split(X, y, groups):
        clf.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        y_true.extend(y.iloc[te_idx].tolist())
        y_pred.extend(clf.predict(X.iloc[te_idx]).tolist())
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    rec_b = recall_score(np.array(y_true) == 'blocked', np.array(y_pred) == 'blocked')
    print(f'{name} -> Acc: {acc:.4f}, Macro F1: {f1:.4f}, Blocked Recall: {rec_b:.4f}')


In [ ]:
# Evaluate Soft-Voting Ensemble
rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42)
hgb = HistGradientBoostingClassifier(max_iter=100, max_depth=6, class_weight='balanced', random_state=42)
ens = VotingClassifier(estimators=[('rf', rf), ('hgb', hgb)], voting='soft')

y_true, y_pred = [], []
for tr_idx, te_idx in logo.split(X, y, groups):
    ens.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    y_true.extend(y.iloc[te_idx].tolist())
    y_pred.extend(ens.predict(X.iloc[te_idx]).tolist())

print('--- Soft-Voting Ensemble (Selected) ---')
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1: {f1_score(y_true, y_pred, average="macro"):.4f}')
